# 🎯 [K15_M08] 검색 질문 형식 통일하기
### 실습 솔루션 & 정답 해설 (Google Colab / Jupyter)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/K15/k15_m08_solution.ipynb)

---

## 📌 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 / 과제** | 군수시설·자산관리 (Level 1) / K15 군 시설 정보 검색 |
| **미션** | 미션 08. 검색 질문 형식 통일하기 (소요시간: 약 1.5시간) |
| **실습 유형** | 조건 제어 프롬프트 엔지니어링 및 팩트체크 검증 (HITL) |

> **⚠️ [교육용 가상 자료 원칙]**
> 본 실습의 모든 자료는 안전 규정에 따라 **교육용 가상 부대 자료(가상 한빛대대)**로 구성되어 있습니다.
> 실제 군사 기밀이나 부대 세부 인적·작전 정보는 외부 AI 도구에 절대 입력하지 않습니다.


In [ ]:
# ============================================================================
# [Colab 환경 설정] 데이터 분석 및 프롬프트 테스트 환경 준비
# ============================================================================
!pip install -q pandas numpy tabulate

import pandas as pd
import numpy as np
import json
import re

# 가상 한빛대대 데이터셋 인메모리 로드 (Colab 환경 독립 실행 보장)
schedule_raw = [
    {"facility": "대대 강당", "day": "월", "09-11": "1중대 교육", "11-13": "1중대 교육", "14-16": "훈련 평가", "16-18": "빈", "19-21": "빈"},
    {"facility": "대대 강당", "day": "화", "09-11": "2중대 교육", "11-13": "빈", "14-16": "3중대 교육", "16-18": "빈", "19-21": "합창 동아리"},
    {"facility": "대대 강당", "day": "수", "09-11": "3중대 교육", "11-13": "3중대 교육", "14-16": "빈", "16-18": "정훈 방송", "19-21": "빈"},
    {"facility": "대대 강당", "day": "목", "09-11": "1중대 안전교육", "11-13": "빈", "14-16": "2중대 교육", "16-18": "2중대 교육", "19-21": "빈"},
    {"facility": "대대 강당", "day": "금", "09-11": "월례 조회", "11-13": "점검(음향)", "14-16": "빈", "16-18": "빈", "19-21": "영화 상영"},
    {"facility": "대대 강당", "day": "토", "09-11": "빈", "11-13": "빈", "14-16": "빈", "16-18": "빈", "19-21": "영화 상영"},
    {"facility": "대대 강당", "day": "일", "09-11": "빈", "11-13": "빈", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 A", "day": "월", "09-11": "분대장 회의", "11-13": "빈", "14-16": "분대장 회의", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 A", "day": "화", "09-11": "빈", "11-13": "인사과 상담", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 A", "day": "수", "09-11": "빈", "11-13": "빈", "14-16": "1중대 토의", "16-18": "1중대 토의", "19-21": "빈"},
    {"facility": "세미나실 A", "day": "목", "09-11": "2중대 토의", "11-13": "2중대 토의", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 A", "day": "금", "09-11": "빈", "11-13": "빈", "14-16": "3중대 토의", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 A", "day": "토", "09-11": "독서 모임", "11-13": "빈", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 A", "day": "일", "09-11": "빈", "11-13": "빈", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 B", "day": "월", "09-11": "빈", "11-13": "교육 준비", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 B", "day": "화", "09-11": "점검(빔프로젝터)", "11-13": "점검(빔프로젝터)", "14-16": "점검(빔프로젝터)", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 B", "day": "수", "09-11": "3중대 토의", "11-13": "빈", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 B", "day": "목", "09-11": "빈", "11-13": "빈", "14-16": "빈", "16-18": "AI 활용 스터디", "19-21": "AI 활용 스터디"},
    {"facility": "세미나실 B", "day": "금", "09-11": "대대 간부 회의", "11-13": "대대 간부 회의", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 B", "day": "토", "09-11": "빈", "11-13": "자격증 스터디", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 B", "day": "일", "09-11": "빈", "11-13": "빈", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "체력단련실(단체)", "day": "월", "09-11": "체력검정 연습(1·2중대)", "11-13": "빈", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "체력단련실(단체)", "day": "화", "09-11": "체력검정 연습(3중대)", "11-13": "빈", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "체력단련실(단체)", "day": "수", "09-11": "체력검정 연습(1·2중대)", "11-13": "빈", "14-16": "소독(이용 불가)", "16-18": "빈", "19-21": "빈"},
    {"facility": "체력단련실(단체)", "day": "목", "09-11": "체력검정 연습(3중대)", "11-13": "빈", "14-16": "빈", "16-18": "빈", "19-21": "3중대 단체 운동"},
    {"facility": "체력단련실(단체)", "day": "금", "09-11": "체력검정 연습(1·2중대)", "11-13": "빈", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "체력단련실(단체)", "day": "토", "09-11": "빈", "11-13": "빈", "14-16": "대대 체육 연습", "16-18": "대대 체육 연습", "19-21": "빈"},
    {"facility": "체력단련실(단체)", "day": "일", "09-11": "빈", "11-13": "빈", "14-16": "빈", "16-18": "빈", "19-21": "빈"}
]

maintenance_raw = [
    {"id": "H01", "date": "08-03", "facility": "세미나실 B", "type": "고장신고", "content": "빔프로젝터 화면 깜빡임", "dept": "군수과", "status": "완료", "days": 2},
    {"id": "H02", "date": "08-10", "facility": "대대 강당", "type": "정기점검", "content": "음향 설비 점검", "dept": "군수과", "status": "완료", "days": 1},
    {"id": "H03", "date": "08-17", "facility": "체력단련실", "type": "고장신고", "content": "러닝머신 1호 벨트 이탈", "dept": "군수과", "status": "완료", "days": 4},
    {"id": "H04", "date": "08-24", "facility": "세미나실 A", "type": "고장신고", "content": "난방기 소음", "dept": "군수과", "status": "완료", "days": 3},
    {"id": "H05", "date": "08-31", "facility": "대대 강당", "type": "고장신고", "content": "무선 마이크 수신 불량", "dept": "정보통신실", "status": "완료", "days": 2},
    {"id": "H06", "date": "09-02", "facility": "세미나실 B", "type": "고장신고", "content": "빔프로젝터 전원 꺼짐", "dept": "군수과", "status": "완료", "days": 3},
    {"id": "H07", "date": "09-07", "facility": "체력단련실", "type": "정기점검", "content": "운동 기구 안전 점검", "dept": "군수과", "status": "완료", "days": 1},
    {"id": "H08", "date": "09-09", "facility": "사이버지식정보방", "type": "고장신고", "content": "PC 3대 부팅 불가", "dept": "정보통신실", "status": "완료", "days": 2},
    {"id": "H09", "date": "09-14", "facility": "대대 강당", "type": "고장신고", "content": "출입문 잠금장치 불량", "dept": "군수과", "status": "완료", "days": 5},
    {"id": "H10", "date": "09-16", "facility": "세미나실 A", "type": "정기점검", "content": "환기 설비 점검", "dept": "군수과", "status": "완료", "days": 1},
    {"id": "H11", "date": "09-21", "facility": "체력단련실", "type": "고장신고", "content": "러닝머신 1호 벨트 이탈", "dept": "군수과", "status": "완료", "days": 6},
    {"id": "H12", "date": "09-23", "facility": "세미나실 B", "type": "정기점검", "content": "영상 장비 점검", "dept": "군수과", "status": "완료", "days": 1},
    {"id": "H13", "date": "09-28", "facility": "대대 강당", "type": "고장신고", "content": "무선 마이크 수신 불량", "dept": "정보통신실", "status": "완료", "days": 3},
    {"id": "H14", "date": "10-01", "facility": "세미나실 A", "type": "고장신고", "content": "난방기 소음", "dept": "군수과", "status": "완료", "days": 4},
    {"id": "H15", "date": "10-05", "facility": "사이버지식정보방", "type": "정기점검", "content": "PC·네트워크 점검", "dept": "정보통신실", "status": "완료", "days": 1},
    {"id": "H16", "date": "10-08", "facility": "세미나실 B", "type": "고장신고", "content": "빔프로젝터 화면 깜빡임", "dept": "군수과", "status": "완료", "days": 4},
    {"id": "H17", "date": "10-13", "facility": "체력단련실", "type": "고장신고", "content": "샤워실 배수 불량", "dept": "군수과", "status": "완료", "days": 2},
    {"id": "H18", "date": "10-15", "facility": "대대 강당", "type": "정기점검", "content": "조명·음향 점검", "dept": "군수과", "status": "완료", "days": 1},
    {"id": "H19", "date": "10-19", "facility": "세미나실 B", "type": "고장신고", "content": "에어컨 냉기 약함", "dept": "군수과", "status": "처리 중", "days": np.nan},
    {"id": "H20", "date": "10-21", "facility": "체력단련실", "type": "고장신고", "content": "러닝머신 1호 벨트 이탈", "dept": "군수과", "status": "처리 중", "days": np.nan},
    {"id": "H21", "date": "10-22", "facility": "대대 강당", "type": "고장신고", "content": "무선 마이크 수신 불량", "dept": "정보통신실", "status": "접수", "days": np.nan},
    {"id": "H22", "date": "10-23", "facility": "세미나실 A", "type": "정기점검", "content": "난방기 점검", "dept": "군수과", "status": "완료", "days": 1}
]

df_schedule = pd.DataFrame(schedule_raw)
df_maintenance = pd.DataFrame(maintenance_raw)

print("✅ [K15] 가상 한빛대대 실습 환경 및 검산용 데이터프레임 로드 완료!")
print(f"- 1주 시설 예약 현황표: {len(df_schedule)}행 (4개 시설 × 7일)")
print(f"- 시설 점검 및 고장 이력: {len(df_maintenance)}건 (정기점검 7건, 고장신고 15건)")


---

## 🎯 미션 08. 검색 질문 형식 통일하기
**소요시간**: 약 1.5시간 | **난이도**: 초급 (Level 1) | **유형**: AI 활용형

### 📋 미션 개요 & 핵심 학습 목표
**[미션 개요]**
<p>부대원 누구나 일관되고 신뢰성 높은 답변을 얻을 수 있도록 질의 및 응답 표준 서식을 설계합니다. 시설·요일·시간대를 필수로 포함하는 표준 질문 템플릿과 '결과·근거 행·주의사항·문의처'로 구성된 표준 답변 양식을 정의하고, 비정형 구어체 질문 입력 시에도 규격화된 답변이 도출되는지 검증합니다. 실무에서 즉시 활용 가능한 시스템 인터페이스의 기초를 완성합니다.</p>

**[학습 목표 및 체득 역량]**
<p>필수 항목(시설·요일·시간)이 포함된 질문 규격과 표준 4단 답변 서식(결과·근거·주의·문의)을 수립하고, 비정형 구어 질의에도 일관된 출력이 나오는지 테스트합니다. <strong>이를 통해 사용자 입력의 변동성에도 흔들리지 않고 규격화된 행정 출력을 보장하는 입출력 템플릿 표준화 설계 역량을 익힙니다.</strong></p>

### 🛠️ 세부 실습 활동 (GuideSteps)
#### 🔹 Step 1: 질문 틀(시설/요일/시간대)을 정한다.
<h4>이 단계의 역할</h4><p>모호한 입력을 방지하기 위해 사용자가 질의할 때 반드시 포함해야 하는 '표준 질문 템플릿(시설/요일/시간대)' 규격을 정의합니다.</p>
<h4>해야 할 일</h4><ol>
<li><p>필수 입력 3요소인 [시설명], [이용 요일], [희망 시간대(24시 표기)]를 정의합니다.</p></li>
<li><p>사용자가 복사하여 조건만 채워 넣을 수 있는 '표준 질문 양식'을 설계합니다.</p></li>
<li><p>필수 요소가 누락되었을 때(예: "강당 언제 비어요?"처럼 요일 누락) AI가 역으로 되물어 조건을 완성하게 하는 보완 규칙을 구상합니다.</p></li>
</ol>
<h4>💡 표준 질문 템플릿 정의 예시</h4><blockquote><p><strong>[질문 규격 서식]:</strong><br>[시설 검색 표준 질문 서식]<br>- 시설명: (대대 강당 / 세미나실 A / 세미나실 B / 체력단련실)<br>- 희망 요일: (월~일 중 택1)<br>- 희망 시간대: (09-11 / 11-13 / 14-16 / 16-18 / 19-21)<br>- 사용 목적: (교육, 토의, 운동 등)</p></blockquote>
<h4>🔍 검증 포인트 (원문 대조 및 팩트체크)</h4><ul>
<li><strong>3대 필수 요소 포함:</strong> 시설, 요일, 시간대가 모두 필수로 정의되었는가?</li>
<li><strong>실무 편의성:</strong> 장병들이 10초 이내에 작성할 수 있을 만큼 직관적인가?</li>
</ul>
<h4>핵심 확인 지표</h4><blockquote><p>표준 질문 서식 규격안 정의 완료</p></blockquote>
<h4>학습자가 제출해야 하는 내용</h4><ul><li><p>표준 질문 템플릿 서식</p></li></ul>

#### 🔹 Step 2: 답변 형식을 정한다.
<h4>이 단계의 역할</h4><p>AI가 답변을 내놓을 때 군 행정 보고에 적합하도록 '결과 - 근거 행 - 주의사항 - 문의처' 4단계 표준 답변 서식을 설계하고 프롬프트에 강제합니다.</p>
<h4>해야 할 일</h4><ol>
<li><p>표준 답변 4단 구조(1. 검색 결과 / 2. 데이터 근거 행 / 3. 주의사항 및 점검 상태 / 4. 문의 및 예약 안내)를 정의합니다.</p></li>
<li><p>프롬프트에 출력 서식 가이드라인을 마크다운 양식으로 지정합니다.</p></li>
<li><p>규격화된 답변이 줄글 서술형 답변에 비해 가독성과 검증 편의성을 얼마나 향상시키는지 확인합니다.</p></li>
</ol>
<h4>💡 표준 답변 서식 지침 예시</h4><blockquote><p><strong>[답변 출력 규칙 지침]:</strong> "[프롬프트 지침에 추가]<br>※ 모든 답변은 반드시 아래 4단 서식을 엄격히 준수하라.<br>1. [가용 여부]: 이용 가능 / 이용 불가 (해당 시간대)<br>2. [근거 행]: [시설명] | [요일] | [시간대] | [표 표기 상태]<br>3. [주의 사항]: 점검 이력 또는 시설 특이사항 (없으면 '특이사항 없음')<br>4. [안내]: 실제 예약 승인은 대대 행정과(내선 1101) 유선 문의"</p></blockquote>
<h4>🔍 검증 포인트 (원문 대조 및 팩트체크)</h4><ul>
<li><strong>4단 구조 준수:</strong> 1(결과), 2(근거), 3(주의), 4(문의) 항목이 누락 없이 강제되었는가?</li>
<li><strong>가독성 확보:</strong> 모바일이나 인트라넷 화면에서 한눈에 정보 파악이 가능한가?</li>
</ul>
<h4>핵심 확인 지표</h4><blockquote><p>4단 표준 답변 서식 프롬프트 탑재 완료</p></blockquote>
<h4>학습자가 제출해야 하는 내용</h4><ul><li><p>표준 답변 서식 정의서 및 프롬프트 반영본</p></li></ul>

#### 🔹 Step 3: 반말·줄임말 질문으로 시험한다.
<h4>이 단계의 역할</h4><p>사용자가 표준 서식을 지키지 않고 반말이나 줄임말(예: "수욜 저녁 강당 비었냐?")로 질의하더라도, AI가 동일한 4단 표준 서식으로 정중하고 일관되게 답변하는지 시험합니다.</p>
<h4>해야 할 일</h4><ol>
<li><p>비정형 구어체 질문 3종("수욜 저녁 강당 비었냐?", "목욜 오후 세미나실 빈곳좀", "체단실 수욜 2시 가능?")을 입력합니다.</p></li>
<li><p>질문의 거친 어조에 영향받지 않고 정형화된 표준 답변 4단 서식을 그대로 유지하는지 확인합니다.</p></li>
<li><p>줄임말(수욜->수요일, 체단실->체력단련실, 2시->14-16)을 올바른 정식 명칭과 시간대로 매핑했는지 검토합니다.</p></li>
</ol>
<h4>💡 AI 구어체 대응 테스트 예시</h4><blockquote><p><strong>[구어체/줄임말 질의 테스트]:</strong> "[표준화 프롬프트 적용 후]<br>[질문]: 수욜 저녁 강당 비었냐? 비었으면 지금 쓰게."</p></blockquote>
<h4>🔍 검증 포인트 (원문 대조 및 팩트체크)</h4><ul>
<li><strong>서식 불변성:</strong> 구어체 질문에도 불구하고 4단 표준 답변 서식이 100% 유지되었는가?</li>
<li><strong>단어 정규화:</strong> '수욜 저녁 강당'을 '대대 강당 | 수요일 | 19-21'로 정확히 변환하여 검색했는가?</li>
</ul>
<h4>핵심 확인 지표</h4><blockquote><p>비정형 질의 3종에 대한 표준 서식 답변 일치율 100%</p></blockquote>
<h4>학습자가 제출해야 하는 내용</h4><ul><li><p>비정형 질의 시험 내역 및 AI 출력 결과</p></li></ul>

#### 🔹 Step 4: 심화 도전 과제 및 최적화
<h4>이 단계의 역할</h4><p>동일한 검색 결과를 텍스트 줄글이 아닌 '마크다운 표(Table)' 형식으로 출력하도록 요구하여 서식 적응성을 비교하는 심화 도전을 진행합니다.</p>
<h4>해야 할 일</h4><ol>
<li><p>답변 형식을 '마크다운 테이블(상태, 시설, 요일, 시간대, 근거, 문의처)'로 전환하여 질의합니다.</p></li>
<li><p>텍스트 4단 서식과 마크다운 표 서식의 정보 전달력과 화면 가독성을 비교합니다.</p></li>
<li><p>부대 게시판 공지용이나 인트라넷 보고용으로 어떤 형식이 적합한지 사용처를 분류합니다.</p></li>
</ol>
<h4>💡 AI 표 형식 전환 질의 예시</h4><blockquote><p><strong>[표 서식 전환 질의]:</strong> "목요일 오후 14시 이후 세미나실 A와 B의 가용 여부를 텍스트 대신 '시설명 | 요일 | 시간대 | 가용여부 | 점검이력 | 예약문의' 컬럼을 가진 마크다운 테이블로 출력해 줘."</p></blockquote>
<h4>🔍 검증 포인트 (원문 대조 및 팩트체크)</h4><ul>
<li><strong>표 레이아웃:</strong> 컬럼 정렬이 깨지지 않고 표 형식으로 온전하게 출력되었는가?</li>
<li><strong>내용 일치:</strong> 서식이 바뀌어도 검색된 사실관계(세미나실 A 14-16 빈 상태 등)가 동일한가?</li>
</ul>
<h4>핵심 확인 지표</h4><blockquote><p>표 형식 답변 출력 성공 및 서식 비교 메모</p></blockquote>
<h4>학습자가 제출해야 하는 내용</h4><ul><li><p>심화 표 서식 결과물 및 가독성 비교 메모</p></li></ul>

### 📑 실습 작성 서식 및 체크리스트 (Sample Template)

## 1. 표준 질문 템플릿
```
[시설 검색 질의 서식]
- 시설명: 대대 강당 / 세미나실 A / 세미나실 B / 체력단련실
- 희망 요일: 월 / 화 / 수 / 목 / 금 / 토 / 일
- 희망 시간대: 09-11 / 11-13 / 14-16 / 16-18 / 19-21
- 사용 목적: 교육 / 회의 / 동아리 / 체육 등
```

## 2. 4단 표준 답변 서식 지침
```markdown
모든 답변은 반드시 다음 4개 항목으로만 구조화하여 출력하라:
1. [가용 여부]: 이용 가능 / 이용 불가 (해당 시간대 명시)
2. [근거 행]: [시설명] | [요일] | [시간대] | [표 표기 상태]
3. [주의 사항]: 점검 이력 또는 시설 특이사항 (없으면 '특이사항 없음')
4. [문의 안내]: 실제 예약 승인은 대대 행정과(내선 1101) 유선 문의
```

## 3. 비정형 구어체 질문 시험 결과
| 비정형 질문 | 정규화된 조건 인식 | AI 답변의 4단 서식 유지 여부 |
| :--- | :--- | :--- |
| "수욜 저녁 강당 비었냐?" | 대대 강당 \| 수요일 \| 19-21 | [x] 4단 서식 및 공손한 어조 100% 유지 |
| "목욜 오후 세미나실 빈곳좀" | 세미나실 A·B \| 목요일 \| 14시 이후 | [x] 4단 서식 및 세미나실 A/B 구분 출력 |
| "체단실 수욜 2시 가능?" | 체력단련실 \| 수요일 \| 14-16 | [x] 4단 서식 및 '소독으로 이용불가' 안내 |

### 🔍 평가 기준 (PASS 조건)
<h4>미션 판정</h4><p><strong>평가 구분:</strong> 필수</p><p><strong>핵심 평가 기준:</strong> [필수] 입출력 표준화·서식 통일 — 표준 질문 템플릿과 4단 표준 답변 서식을 설계하고 비정형 질의에서도 일관된 출력을 보장했는가? (PASS / FAIL)</p><h4>미션 PASS 조건</h4><ul><li><p>3대 필수 요소가 포함된 표준 질문 템플릿 규격서 완성</p></li><li><p>4단 표준 답변 서식(가용여부·근거행·주의사항·문의처) 프롬프트 반영</p></li><li><p>비정형 구어체 질문 3종 시험 시 표준 서식 준수율 100% 확보</p></li></ul><h4>채점자 판정 메모</h4><blockquote><p>사용자 입력 어조가 거칠거나 축약형이더라도 AI 답변은 언제나 정형화된 행정 보고 규격을 변함없이 유지하는지 확인한다.</p></blockquote>


In [ ]:
# [미션 08] 프롬프트 및 실습 결과 기록
mission_08_prompt = """# [미션 08] 입출력 표준화 및 비정형 구어체 질문 정규화 질의
너는 가상 한빛대대 시설 정보 검색 도우미다.
사용자가 반말, 줄임말, 비정형 구어로 질문하더라도 의도를 정규화하여 다음 4단 표준 서식으로만 답변하라:

[4단 표준 답변 서식]
1. [가용 여부]: 이용 가능 / 이용 불가 (해당 요일/시간대 명시)
2. [근거 행]: [시설명] | [요일] | [시간대] | [표 표기 상태]
3. [주의 사항]: 점검 이력 또는 시설 특이사항 (없으면 '특이사항 없음')
4. [문의 안내]: 실제 예약 승인은 대대 행정과(내선 1101) 유선 문의

[비정형 테스트 질문 3건]
1. "수욜 저녁 강당 비었냐?"
2. "목욜 오후 세미나실 빈곳좀"
3. "체단실 수욜 2시 가능?"
"""

mission_08_result = """### 📋 비정형 구어체 질문 시험 및 4단 표준 답변 결과

#### 1. 비정형 구어체 질문 정규화 시험 결과표
| 비정형 질문 | 정규화된 조건 인식 | AI 답변의 4단 서식 준수 여부 | 판정 |
| :--- | :--- | :--- | :---: |
| "수욜 저녁 강당 비었냐?" | 대대 강당 \| 수요일 \| 19-21 | 1. [가용 여부]: 이용 가능 (수요일 19-21시)<br>2. [근거 행]: 대대 강당 \| 수 \| 19-21 \| 빈<br>3. [주의 사항]: 특이사항 없음<br>4. [문의 안내]: 행정과(내선 1101) 문의 | **PASS** |
| "목욜 오후 세미나실 빈곳좀" | 세미나실 A·B \| 목요일 \| 14시 이후 | 1. [가용 여부]: 세미나실 A(14-18), B(14-16) 가능<br>2. [근거 행]: 세미나실 A·B \| 목 \| 14-16/16-18 \| 빈<br>3. [주의 사항]: 세미나실 B 16시 이후 AI스터디<br>4. [문의 안내]: 행정과(내선 1101) 문의 | **PASS** |
| "체단실 수욜 2시 가능?" | 체력단련실 \| 수요일 \| 14-16 | 1. [가용 여부]: 이용 불가 (수요일 14-16시)<br>2. [근거 행]: 체력단련실(단체) \| 수 \| 14-16 \| 소독<br>3. [주의 사항]: 방역 소독으로 이용 불가<br>4. [문의 안내]: 행정과(내선 1101) 문의 | **PASS** |

- [x] 비정형 구어 질의에도 4단 서식 100% 준수 및 공손한 행정 어조 유지 확인.
"""

print(f"미션 08 실습 기록 완료: 프롬프트 {len(mission_08_prompt.strip())}자, 결과 {len(mission_08_result.strip())}자")


In [ ]:
# [데이터 교차 검산 - 미션8] 4단 표준 답변 서식 정규성 검증 함수
def validate_4step_format(answer_text):
    step1 = "1. [가용 여부]" in answer_text or "[가용 여부]" in answer_text
    step2 = "2. [근거 행]" in answer_text or "[근거 행]" in answer_text
    step3 = "3. [주의 사항]" in answer_text or "[주의 사항]" in answer_text
    step4 = "4. [문의 안내]" in answer_text or "[문의 안내]" in answer_text
    is_valid = step1 and step2 and step3 and step4
    return {"4단 서식 준수 여부": is_valid, "항목별": [step1, step2, step3, step4]}

sample_ans = """
1. [가용 여부]: 수요일 19-21 이용 가능
2. [근거 행]: 대대 강당 | 수요일 | 19-21 | 상태: 빈
3. [주의 사항]: 특이사항 없음
4. [문의 안내]: 행정과(1101) 유선 문의
"""
print("서식 유효성 검사 결과:", validate_4step_format(sample_ans))
